# Exploratory Data Analysis

Your first task is to explore the dataset you’ve been assigned. It’s your responsibility to complete all the to-do steps and generate the following:

- Identify the `type of dataset`
- Print the `list of building IDs`
- Select 3 random building IDs, and for a year within its available data range, determine and print the following: 1) `Start date` 2) `End date` 3) `Latitude` 4) `Longitude` 5) `Building type` 6) `Plot the data from day 1 for the selected buildings`

__Important__: For this exercise, start by scrolling to the last cell and completing the first three `TODO` tasks. When you need to update the `plot_load` function, go to the cell above, make the necessary changes, and then return to the last cell to continue.`

__TIP__: Please refer to the table below to view the data ranges and example building IDs for each dataset.

```
| Dataset    | Data Range  | Example ID     |
|------------|-------------|----------------|
| Ideal      | 2017–2018   | home241        |
| Electricity| 2012–2014   | MT_061         |
| lcl        | 2012–2013   | MAC003538      |
| borealis   | 2011        | home25         |
```

Below is a screenshot of the output I generated. Your output should look similar. Feel free to modify the plot or adjust other features for your convenience, but only make changes where you see `# TODO:`. Do not edit any other sections of the code.

![NN model architecture](../../Images/EDA.png)

In [ ]:
# ------------------- #
# --- Do Not Edit --- #
# ------------------- #

import os
from pathlib import Path
from buildings_bench import load_torch_dataset
from buildings_bench.transforms import TimestampTransform
import matplotlib.pyplot as plt
import numpy as np

os.environ["BUILDINGS_BENCH"] = "/global/cfs/cdirs/m4388/2025_Bootcamp/Project4/Dataset"

class EDA:
    """
    Exploratory Data Analysis (EDA) helper for a single BuildingsBench
    real-world dataset (e.g. 'ideal', 'electricity', 'lcl', 'sceaux', 'borealis').

    Wraps the object returned by load_torch_dataset(dataset_name, ...) and
    exposes convenience methods for inspecting and plotting individual buildings.

    Usage:
        dataset = load_torch_dataset('ideal', apply_scaler_transform='')
        eda = EDA('ideal', dataset)              # pass the RAW loader output, not .building_datasets
        eda.dataset_summary()                    # dataset type + list of building IDs
        eda.building_summary('home241', 2018)    # start/end date, lat/lon, building type
        eda.plot_load('home241', 2018, day=1)    # plot the load curve for one day-index window

    Attributes:
        name (str): Dataset name, for display only.
        datasets (dict): dataset.building_datasets -- maps building_id to a
            list of (year, TorchBuildingDataset) tuples.

    Note: each TorchBuildingDataset item (ds[year][day]) returns a
    context_len + pred_len window (168 + 24 hours = 8 days by default),
    not a single 24-hour day -- see the "one day" fix from the other notebook
    if you want plot_load's `day` argument to mean literally one calendar day.
    """
    def __init__(self, dataset_name, dataset):
        self.name = dataset_name
        self.datasets = dataset.building_datasets

    @staticmethod
    def reconstruct_datetime(data, year):
        """Convert a sample's day_of_year/day_of_week/hour_of_day arrays back
        into real numpy datetime64 timestamps for the given calendar year."""
        timestamp = np.concatenate([
            data['day_of_year'],
            data['day_of_week'],
            data['hour_of_day']
        ], axis=-1)

        timestamp = TimestampTransform().undo_transform(timestamp)

        dt = (
            np.datetime64(f'{year}-01-01')
            + np.timedelta64(1, 'D') * (timestamp[..., 0] - 1)
            + np.timedelta64(1, 'h') * timestamp[..., -1]
        )

        return dt, timestamp

    def dataset_summary(self):
        """Print the dataset name, the type of `self.datasets`, and all building IDs."""
        print(f"Dataset: {self.name}")
        print(f"Type: {type(self.datasets)}")
        print(f"Building IDs: {list(self.datasets.keys())}")

    def building_summary(self, building_id, year):
        """Print start/end datetime, latitude, longitude, and building type
        for one building in one year."""
        ds = dict(self.datasets[building_id])

        data_start = ds[year][0]
        data_end = ds[year][len(ds[year])-1]

        dt_start, _ = self.reconstruct_datetime(data_start, year)
        dt_end, _ = self.reconstruct_datetime(data_end, year)

        lat = float(data_start['latitude'][0][0])
        lon = float(data_start['longitude'][0][0])
        btype = int(data_start['building_type'][0][0])

        print(f"Building ID: {building_id}")
        print(f"Year: {year}")
        print(f"Number of records in {building_id}: {len(ds[year])}")
        print(f"Start date: {dt_start[0]}")
        print(f"End date: {dt_end[-1]}")
        print(f"Latitude: {lat:.4f}")
        print(f"Longitude: {lon:.4f}")
        print(f"Building Type: {self.building_type_str(btype)}")

    @staticmethod
    def building_type_str(building_type):
        """Convert a building_type integer (0 or 1) into a human-readable label.
    
        0 = Residential, 1 = Commercial (buildings_bench.BuildingTypes.RESIDENTIAL_INT/COMMERCIAL_INT).
        """
        return {0: 'Residential', 1: 'Commercial'}.get(building_type, f'Unknown ({building_type})')
        
    def plot_load(self, building_id, year, day):
        """Plot the load curve for one building/year/day-index window."""
        ds = dict(self.datasets[building_id])

        if day < 0 or day >= len(ds[year]):
            print(f"Invalid day index: {day}. Valid range is 0 to {len(ds[year])-1}.")
            return

        data = ds[year][day]
        dt, _ = self.reconstruct_datetime(data, year)
        load = data['load'].squeeze()

        lat = float(data['latitude'][0][0])
        lon = float(data['longitude'][0][0])
        btype = int(data['building_type'][0][0])

        plt.figure(figsize=(12, 4))
        plt.plot(dt.astype('datetime64[m]'), load, color='blue', linewidth=1)
        plt.xlabel('Datetime', fontsize=14)
        plt.ylabel('Load (kW)', fontsize=14)
        plt.title(
            f'Building: {building_id}  |  Year: {year}  |  Day: {day}\n'
            f'Lat: {lat:.4f}, Lon: {lon:.4f}, Building Type: {self.building_type_str(btype)}',
            fontsize=13
        )
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.grid(True)
        plt.show()

print('Finished')


In [ ]:
# ------------------- #
# ------ EDIT ------- #
# ------------------- #

import random

if __name__ == "__main__":
    dataset_name = #TODO: pick one of: 'ideal', 'electricity', 'lcl', 'borealis'
    dataset = load_torch_dataset(dataset_name, apply_scaler_transform='')

    #TODO: Instantiate the EDA class

    #TODO: Run dataset_summary method

    #TODO: Select 3 random building_ids and run building_summary for a valid year
    building_ids = 
    selected_ids = 

    for building_id in selected_ids:
        #TODO: a building can have data for more than one year -- pick one that's available
        available_years = [year for year, _ in eda.datasets[building_id]]
        year = random.choice(available_years)

        #TODO: Run dataset_summary method for the given year

        #TODO: Plot day 1 for this building

# ------------------- #
# ------ EDIT ------- #
# ------------------- #


## Next Step:

Think about ways that you could present this information to someone who does not know anything about this project or dataset. This will be important for your final presentation!

The next step will be:
`/BuildingsBenchTutorial/Tutorials/Final-Project-Modules/Train-Model.ipynb`